# 学習データの中の付け間違えたラベルを見つける (Confident Learning)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_classification/label_noise.ipynb)

NLP クックブック 11-18 ／ テキスト分類

学習データに紛れ込んだラベルの誤りを、交差検証の予測確率から自動で洗い出したい。livedoor ニュースコーパスにわざと誤りを入れて、どれだけ見つかるか、除くと精度がどう変わるかを確かめる。

In [1]:
!pip install -q fugashi unidic-lite cleanlab

In [2]:
import io, tarfile, urllib.request
import numpy as np
import pandas as pd
from fugashi import Tagger
from sklearn.model_selection import train_test_split

def load_livedoor():
    """livedoor ニュースコーパス (CC BY-ND 2.1 JP) を DataFrame にする"""
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
tagger = Tagger("-Owakati")
df["wakati"] = [tagger.parse(t[:2000]).strip() for t in df["text"]]   # Colab の CPU で 1〜2 分
names = sorted(df["label"].unique())
train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=0)
train = train.reset_index(drop=True)
y_true = train["label"].map(names.index).to_numpy()   # 元のラベル (整数 ID)
y_test = test["label"].map(names.index).to_numpy()
print(f"学習 {len(train)} 件 / テスト {len(test)} 件")

学習 5893 件 / テスト 1474 件


In [3]:
# まず、同じタイトルの記事が別のラベルで二重に入っていないかを確かめる (矛盾するラベルの典型)
# 記事: livedoor ニュースの各配信元。収集・配布: 株式会社ロンウイット (livedoor ニュースコーパス、CC BY-ND 2.1 JP)
df[df.groupby("title")["label"].transform("nunique") > 1][["label", "title"]]

,label,title
2516,smax,iPhone女子部がフリーペーパー「iPhone♡」を大手量販店やソフトバンクショップで配布開始
5622,kaden-channel,iPhone女子部がフリーペーパー「iPhone♡」を大手量販店やソフトバンクショップで配布開始


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

def make_clf():
    return make_pipeline(TfidfVectorizer(token_pattern=r"\S+", min_df=3, max_df=0.5, sublinear_tf=True),
                         LogisticRegression(C=10, max_iter=1000))

cv = StratifiedKFold(5, shuffle=True, random_state=0)
# 付け替え先を決めるため、誤りを入れる前のラベルで交差検証し、各クラスで最も取り違えやすいクラスを求める (Colab の CPU で 2 分程度)
proba_orig = cross_val_predict(make_clf(), train["wakati"], y_true, cv=cv, method="predict_proba", n_jobs=-1)   # 最後のセルでも使う
cm = confusion_matrix(y_true, proba_orig.argmax(axis=1))
np.fill_diagonal(cm, 0)
swap = cm.argmax(axis=1)
print({names[i]: names[j] for i, j in enumerate(swap)})

NOISE_RATE = 0.2   # 誤りの割合。0.1 や 0.4 に変えて試すとよい
rng = np.random.default_rng(0)
flip = rng.random(len(train)) < NOISE_RATE   # 誤りを入れる事例 (テストデータには入れない)
noisy = {
    "系統的": np.where(flip, swap[y_true], y_true),   # 取り違えやすいクラスに付け替える
    "ランダム": np.where(flip, (y_true + rng.integers(1, len(names), len(train))) % len(names), y_true),   # ばらばらなクラスに付け替える
}
print(f"わざと入れた誤り: {flip.sum()} 件 ({NOISE_RATE:.0%})")

{'dokujo-tsushin': 'peachy', 'it-life-hack': 'dokujo-tsushin', 'kaden-channel': 'topic-news', 'livedoor-homme': 'peachy', 'movie-enter': 'peachy', 'peachy': 'dokujo-tsushin', 'smax': 'it-life-hack', 'sports-watch': 'topic-news', 'topic-news': 'sports-watch'}
わざと入れた誤り: 1221 件 (20%)


In [5]:
from cleanlab.filter import find_label_issues

rows, proba, issues = [], {}, {}
for kind, y in noisy.items():
    # 交差検証で、すべての事例について「その事例を学習に使っていないモデル」の予測確率を得る (1 回 2 分程度)
    p = proba[kind] = cross_val_predict(make_clf(), train["wakati"], y, cv=cv, method="predict_proba", n_jobs=-1)
    # (1) 予測と付いたラベルが食い違うものをすべて候補にする (論文の C_confusion。この章の「エラー分析」のやり方)
    c_confusion = np.flatnonzero(p.argmax(axis=1) != y)
    # (2) confident joint の非対角を自分で実装: クラス j のしきい値 = ラベル j が付いた事例での p(j) の平均。
    #     しきい値を超えたクラスのうち確率最大のものが付いたラベルと違えば候補 (filter_by="confident_learning" と同じ考え方)
    thr = np.array([p[y == j, j].mean() for j in range(len(names))])
    over = p >= thr
    cj_offdiag = np.flatnonzero(over.any(axis=1) & (np.where(over, p, -1).argmax(axis=1) != y))
    # (3) cleanlab の既定 (prune by noise rate): confident joint から (付いたラベル, 本当のラベル) の組ごとの誤りの件数を推定し、
    #     組ごとに margin (p(本当のラベル) - p(付いたラベル)) の大きい順にその件数を選ぶ。結果は自信度の低い順に並べる
    issues[kind] = find_label_issues(y, p, filter_by="prune_by_noise_rate", return_indices_ranked_by="self_confidence")
    for method, idx in [("予測 ≠ 付いたラベル (C_confusion)", c_confusion), ("confident joint の非対角 (自作)", cj_offdiag),
                        ("prune by noise rate (cleanlab)", issues[kind])]:
        rows.append({"誤り": kind, "方法": method, "候補数": len(idx),
                     "適合率": flip[idx].mean(), "再現率": flip[idx].sum() / flip.sum()})
pd.DataFrame(rows).round(3)

,誤り,方法,候補数,適合率,再現率
0,系統的,予測 ≠ 付いたラベル (C_confusion),1706,0.655,0.916
1,系統的,confident joint の非対角 (自作),1157,0.764,0.724
2,系統的,prune by noise rate (cleanlab),1413,0.725,0.839
3,ランダム,予測 ≠ 付いたラベル (C_confusion),1554,0.777,0.989
4,ランダム,confident joint の非対角 (自作),988,0.951,0.770
5,ランダム,prune by noise rate (cleanlab),1251,0.909,0.931


In [6]:
boot_rng = np.random.default_rng(0)
def boot_ci(a, b, n=1000):
    """正解の 0/1 の差のブートストラップ 95% 区間。テスト事例の選び方によるばらつきだけを表す (学習の乱数などは含まない)"""
    d = a.astype(float) - b
    s = [d[boot_rng.integers(0, len(d), len(d))].mean() for _ in range(n)]
    return f"{d.mean():+.3f} (95% 区間 {np.percentile(s, 2.5):+.3f} 〜 {np.percentile(s, 97.5):+.3f})"

def fit_eval(idx, y):
    return make_clf().fit(train["wakati"].iloc[idx], y[idx]).predict(test["wakati"]) == y_test

# 5 通り学習してテストする (Colab の CPU で 3〜4 分)
all_idx = np.arange(len(train))
ideal = fit_eval(all_idx, y_true)
print(f"誤りを入れる前: テスト正解率 {ideal.mean():.3f}")
res = {}
for kind, y in noisy.items():
    keep = np.setdiff1d(all_idx, issues[kind])
    res[kind] = fit_eval(all_idx, y), fit_eval(keep, y)
    print(f"[{kind}] 誤りを含んだまま {res[kind][0].mean():.3f} → 候補 {len(issues[kind])} 件を除いて {res[kind][1].mean():.3f}"
          f" (残った誤り {(flip[keep]).sum()} 件 / 除いた正しい事例 {(~flip[issues[kind]]).sum()} 件)")
    print("    除いた効果:", boot_ci(res[kind][1], res[kind][0]))
print("誤りを含んだままの差 (ランダム − 系統的):", boot_ci(res["ランダム"][0], res["系統的"][0]))

誤りを入れる前: テスト正解率 0.949
[系統的] 誤りを含んだまま 0.864 → 候補 1413 件を除いて 0.920 (残った誤り 196 件 / 除いた正しい事例 388 件)
    除いた効果: +0.056 (95% 区間 +0.041 〜 +0.070)
[ランダム] 誤りを含んだまま 0.925 → 候補 1251 件を除いて 0.937 (残った誤り 84 件 / 除いた正しい事例 114 件)
    除いた効果: +0.012 (95% 区間 +0.000 〜 +0.022)
誤りを含んだままの差 (ランダム − 系統的): +0.061 (95% 区間 +0.045 〜 +0.075)


In [7]:
# わざと入れた誤りとは別に、元のデータにもともとある「怪しい」事例を探す (誤りを入れる前のラベルとその交差検証の確率を使う)
# 記事: livedoor ニュースの各配信元。収集・配布: 株式会社ロンウイット (livedoor ニュースコーパス、CC BY-ND 2.1 JP)
pd.set_option("display.max_colwidth", 70)
idx = find_label_issues(y_true, proba_orig, filter_by="prune_by_noise_rate", return_indices_ranked_by="self_confidence")
cand = pd.DataFrame({"title": train["title"], "付いたラベル": [names[i] for i in y_true], "予測": [names[i] for i in proba_orig.argmax(axis=1)],
                     "p(付いたラベル)": proba_orig[np.arange(len(train)), y_true].round(3)}).loc[idx]
print("元のデータの候補:", len(idx), "件")
cand.head(10)

元のデータの候補: 21 件


,title,付いたラベル,予測,p(付いたラベル)
4516,iPhone女子部がフリーペーパー「iPhone♡」を大手量販店やソフトバンクショップで配布開始,kaden-channel,smax,0.003
3432,細マッチョからスーパーマンまで、光り輝く“神”ボディの作り方,livedoor-homme,movie-enter,0.008
5494,阪神が一番嫌いだった！ 手を抜いたことある！ 新庄剛志がテレビで衝撃告白【話題】,kaden-channel,sports-watch,0.009
972,ロバート・デ・ニーロ、塚本晋也監督の「鉄男4」に出演決定？,livedoor-homme,movie-enter,0.015
3235,今年のクリスマス、リムジンのお出迎えでセレブな1日はいかが？,dokujo-tsushin,peachy,0.016
3270,徹底検証！話題のセキュリティソフト「マカフィー オール アクセス 2012」をITニュース編集長が使い倒してみた,livedoor-homme,it-life-hack,0.021
4944,映画『朱花の月』とPHOTOPRESSOがコラボ『万葉のまなざし』,livedoor-homme,movie-enter,0.025
964,【終了しました】夏の紫外線からカラーリングした髪をケアする「カラーコンサーブ デイリー カラー プロテクト」を3名様にプレゼント！,dokujo-tsushin,peachy,0.032
3157,あったかスープで、おなかもココロも満たされるキャンペーン実施中！,dokujo-tsushin,peachy,0.040
4609,iPhone対抗の本命 最新のAndroid「Desire」の実力,livedoor-homme,smax,0.046


- 交差検証の予測確率を使う点は、この章の「モデルが自信満々に間違えた事例を調べる (エラー分析)」と同じ。そこでのやり方にあたる (1) 「予測 ≠ 付いたラベル」は、再現率は高い (系統的 0.916、ランダム 0.989) が、適合率は 0.655 / 0.777 で、系統的では 3 割強、ランダムでは 2 割強の候補がわざと入れた誤りではない。
- Confident Learning は Northcutt, Jiang, Chuang が提案した (arXiv 2019 年、JAIR 70 巻、2021 年)。誤り方が (付いたラベル, 本当のラベル) の組ごとに決まる class-conditional noise を前提に、その件数表 (confident joint) を推定する。論文では (1) を基準となる最も単純な方法 C_confusion、(2) を confident joint の非対角を使う方法、(3) を prune by noise rate と呼ぶ。この実験では (2) は再現率が低め (0.724 / 0.770) で、(3) は適合率と再現率の釣り合いがよかった。`cleanlab` のライセンスは Apache-2.0。
- 実際の付け間違いは似たクラスの取り違えに偏り、手法の前提にも合うので、系統的な誤りを主にした。付け替え先は、誤りを入れる前のデータの交差検証の混同行列で、各クラスが最も間違えられやすいクラスにした (独女通信 ↔ Peachy、Sports Watch ↔ トピックニュースなど)。系統的な誤りはランダムな誤りより見つけにくく (cleanlab の適合率 0.725 対 0.909)、害も大きかった (誤りを含んだままの正解率 0.864 対 0.925)。
- 候補を除いて学習し直すと、系統的な誤りでは 0.864 → 0.920 (+0.056、95% 区間 +0.041〜+0.070) と大きく戻ったが、ランダムな誤りでは 0.925 → 0.937 (+0.012、区間 +0.000〜+0.022) と効果が小さかった。ランダムな誤りは、もともとの害が小さいぶん除いて得られるものも少ない。誤りを入れる前の 0.949 に届かないのは、残った誤り (196 件) と、一緒に除いた正しい事例 (388 件) のためと考えられる。区間はテスト事例のブートストラップで求めたもので (評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」を参照)、テスト事例の選び方によるばらつきだけを表し、1 回の試行の結果である。`NOISE_RATE` を変えて試すとよい。
- 元のデータの候補 (21 件) の最上位は、重複を調べたセルで見つかった記事の kaden-channel 側 (予測 smax) だった。家電チャンネルに入っている新庄剛志の野球の記事 (予測 sports-watch) や、独女通信のプレゼントやキャンペーンの記事 (予測 peachy) も上位にある。このコーパスのラベルは配信元のサイト名なので、多くは付け間違いというより内容が別のサイトに近い境界例である。候補は自動で消さず、人が読んで「直す・除く・そのまま」を決める。重複は完全一致で先に洗い出す。
- 学習に使った事例の確率は過信しているので、必ず交差検証の確率を使う。論文では除いた後にクラスごとに損失を重み付けし直すが、ここでは単に除いて学習し直している。
- この章の「ルールでラベルを付けて分類器を学習する (弱教師あり学習、Snorkel)」で作ったラベルにも誤りが含まれる。このレシピの方法で候補を洗い出して確かめられる。
- livedoor ニュースコーパスは CC BY-ND 2.1 JP (改変禁止)。記事そのものは改変していない (付け替えたのはカテゴリのラベルだけ) が、それでも、ラベルを付け替えたデータを livedoor ニュースコーパスとして配布しない。